<a href="https://colab.research.google.com/github/pujaroy280/DATA612/blob/main/DATA_612_Project_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project 5: Implementing a Recommender System on Spark**

Puja Roy

Summer 2026

## **Introduction**

The purpose of this project is to develop a recommender system using Spark on Python to build onto the recommendation system I previously created for Project 2 using 2 collaborative filtering methods. The MovieLens 100K dataset was used by downloading it from: https://grouplens.org/datasets/movielens/100k/


## **Step 1 - Start Spark Session**

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MovieLens ALS Recommender") \
    .getOrCreate()

spark

## **Step 2 - Load MovieLens 100K Dataset**

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Download and extract the MovieLens 100k dataset
!wget -nc http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -n ml-100k.zip

schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", IntegerType(), True)
])

ratings = spark.read.csv(
    "ml-100k/u.data", # Updated path
    sep="\t",
    schema=schema
)

ratings.show(5)

--2026-07-06 06:41:48--  http://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-100k.zip [following]
--2026-07-06 06:41:48--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  3.53MB/s    in 1.3s    

2026-07-06 06:41:50 (3.53 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]

Archive:  ml-100k.zip
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|   196|    242|     3|881250949|
|   186|    302

## **Step 3 - Explore Basic Data**

In [4]:
ratings.printSchema()
ratings.describe().show()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+------------------+------------------+------------------+-----------------+
|summary|            userId|           movieId|            rating|        timestamp|
+-------+------------------+------------------+------------------+-----------------+
|  count|            100000|            100000|            100000|           100000|
|   mean|         462.48475|         425.53013|           3.52986|8.8352885148862E8|
| stddev|266.61442012750905|330.79835632558473|1.1256735991443214|5343856.189502848|
|    min|                 1|                 1|                 1|        874724710|
|    max|               943|              1682|                 5|        893286638|
+-------+------------------+------------------+------------------+-----------------+



In [5]:
num_ratings = ratings.count()
num_users = ratings.select("userId").distinct().count()
num_movies = ratings.select("movieId").distinct().count()

print("Ratings:", num_ratings)
print("Users:", num_users)
print("Movies:", num_movies)

Ratings: 100000
Users: 943
Movies: 1682


## **Step 4 - Train-Test Split**

In [6]:
train, test = ratings.randomSplit([0.8, 0.2], seed=42)

## **Step 5 - Build ALS Recommender Model (Spark MLlib)**

ALS = Alternating Least Squares (collaborative filtering at scale)

In [7]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    nonnegative=True,
    coldStartStrategy="drop",
    implicitPrefs=False
)

model = als.fit(train)

## **Step 6 - Make Predictions**

In [8]:
predictions = model.transform(test)
predictions.show(5)

+------+-------+------+---------+----------+
|userId|movieId|rating|timestamp|prediction|
+------+-------+------+---------+----------+
|   148|      8|     4|877020297| 4.1538553|
|   148|     56|     5|877398212| 3.7026064|
|   148|     71|     5|877019251|  3.164286|
|   148|    133|     5|877019251| 2.8715222|
|   148|    169|     5|877020297| 4.7239532|
+------+-------+------+---------+----------+
only showing top 5 rows


## **Step 7 - Evaluate Model (RMSE)**

In [9]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print("ALS RMSE:", rmse)

ALS RMSE: 0.9166093730681385


## **Step 8 - Hyperparameter Tuning ALS Model**

In [10]:
als_tuned = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=10,           # latent factors
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

model_tuned = als_tuned.fit(train)
predictions_tuned = model_tuned.transform(test)

rmse_tuned = evaluator.evaluate(predictions_tuned)
print("Tuned ALS RMSE:", rmse_tuned)

Tuned ALS RMSE: 0.9184635541471368


## **Step 9 - Generate Top User and Movie Recommendations**

In [11]:
user_recs = model.recommendForAllUsers(5)
user_recs.show(truncate=False)

+------+----------------------------------------------------------------------------------------------+
|userId|recommendations                                                                               |
+------+----------------------------------------------------------------------------------------------+
|1     |[{1449, 5.137939}, {408, 4.9423304}, {169, 4.9317923}, {114, 4.84188}, {1467, 4.7708263}]     |
|2     |[{1643, 5.209463}, {1449, 5.008093}, {318, 4.773612}, {483, 4.734856}, {1398, 4.712831}]      |
|3     |[{902, 4.464282}, {1612, 4.4266133}, {320, 4.336726}, {1591, 4.3299184}, {1643, 4.2507987}]   |
|4     |[{1449, 6.0246124}, {1512, 5.706402}, {1193, 5.652414}, {1167, 5.5458403}, {1585, 5.5422487}] |
|5     |[{1129, 4.5467744}, {408, 4.471028}, {850, 4.4414372}, {169, 4.4150486}, {114, 4.3701415}]    |
|6     |[{1643, 5.07753}, {694, 4.5694237}, {1405, 4.532691}, {1203, 4.4551497}, {1449, 4.446643}]    |
|7     |[{1643, 6.2918878}, {119, 5.0968733}, {1169, 4.9919705},

In [12]:
movie_recs = model.recommendForAllItems(5)
movie_recs.show(truncate=False)

+-------+-----------------------------------------------------------------------------------------+
|movieId|recommendations                                                                          |
+-------+-----------------------------------------------------------------------------------------+
|1      |[{688, 5.123417}, {849, 5.059278}, {810, 4.9010077}, {939, 4.7597785}, {477, 4.7393494}] |
|3      |[{688, 4.3782663}, {636, 4.3709826}, {628, 4.3466578}, {811, 4.328082}, {859, 4.3193517}]|
|5      |[{688, 5.064862}, {849, 4.682706}, {907, 4.6276116}, {427, 4.601594}, {507, 4.565185}]   |
|6      |[{928, 5.7123}, {341, 5.7071724}, {427, 5.5417376}, {519, 5.4976425}, {688, 5.430249}]   |
|9      |[{928, 4.9535365}, {686, 4.8358474}, {765, 4.8166423}, {98, 4.7263136}, {252, 4.6910214}]|
|12     |[{118, 5.170257}, {810, 5.165238}, {640, 5.052108}, {16, 5.021252}, {849, 5.0001493}]    |
|13     |[{239, 4.376831}, {115, 4.341768}, {34, 4.3298607}, {808, 4.321047}, {68, 4.2268825}]    |


## **Step 10 - Compare Results with Project 2 Results**

Based on the results of the ALS model, the ALS RMSE is 0.9166093730681385 and the fine tuned RMSE is 0.9184635541471368. Based on the previous results for the 4 collaborative filtering models were:

   Algorithm      RMSE
1  User-User Pearson  1.022424
0   User-User Cosine  1.028382
2   Item-Item Cosine  1.046763
3  Item-Item Pearson  1.064462

The results display that Spark ALS is more scalable and has a strong predictive performance. Spark ALS is better since it manages millions of ratings across clusters and does not store full user-user similarity matrix. A distributed platform like Spark would become nescessary is when the dataset exceeds the limit of memory within a single machine such as millions to billions of ratings. Also, it would be recommended to utilize when real time or estimated real time is required. To add on, Spark would come in handy when matrix-based similarity computations become complicated.